# 💬 Modul 06 Hands-on: Applied NLP & Modern Transformers

Di hands-on ini, Anda akan mempraktikkan alur kerja NLP modern menggunakan ekosistem Hugging Face:
1. **Membedah Tokenizer**: Melihat bagaimana teks mentah dipecah menjadi `input_ids` dan `attention_mask`.
2. **Dataset Teks Riil**: Membangun dataset ulasan produk e-commerce.
3. **Fine-Tuning DistilBERT**: Melatih model Transformer Encoder untuk klasifikasi sentimen.
4. **Live Inference Tester**: Menguji model dengan kalimat bebas yang Anda ketik sendiri!

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import numpy as np

device = torch.device("mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu"))
print(f"🚀 Device aktif: {device}")

## 1. Membedah Tokenizer: Dari Karakter ke Tensor
Kita gunakan tokenizer dari model `distilbert-base-uncased`. Amati bagaimana subword tokenization memecah kata.

In [ ]:
MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

sample_text = "Deep learning with transformers is unbelievably amazing!"
encoded = tokenizer(sample_text)

print(f"Teks Asli: {sample_text}")
print("\nHasil Tokenisasi (Subwords):")
tokens = tokenizer.convert_ids_to_tokens(encoded["input_ids"])
print(tokens)

print("\nInput IDs (ID Angka untuk tiap token):")
print(encoded["input_ids"])

print("\nAttention Mask (1 = Token asli, 0 = Padding kosong):")
print(encoded["attention_mask"])
print("\n💡 Perhatikan token khusus: [CLS] di awal (representasi kalimat) dan [SEP] di akhir (pemisah)!")

## 2. Menyiapkan Dataset Ulasan Produk (Sentiment Dataset)
Kita siapkan dataset ulasan pelanggan untuk klasifikasi biner: 1 (Positif) vs 0 (Negatif).

In [ ]:
texts = [
    "This laptop is incredibly fast, battery lasts all day!",
    "Superb build quality and stunning OLED display. Highly recommended!",
    "Excellent customer support, my issue was resolved in ten minutes.",
    "Best purchase I made this year, totally worth every penny.",
    "Great product, sleek design and works seamlessly out of the box.",
    "Terrible experience. The screen arrived cracked and customer service refused refund.",
    "Awful battery life, dies after barely one hour of light usage.",
    "Very disappointed with the quality, feels cheap and sluggish performance.",
    "Do not buy this! It overheated and stopped working on day two.",
    "Complete waste of money. Faulty charger and noisy cooling fan."
] * 10  # Duplikasi untuk demo dataset 100 sampel

labels = [1, 1, 1, 1, 1, 0, 0, 0, 0, 0] * 10

class TextSentimentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=64):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
        
    def __len__(self):
        return len(self.texts)
        
    def __getitem__(self, idx):
        text = self.texts[idx]
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )
        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "label": torch.tensor(self.labels[idx], dtype=torch.long)
        }

dataset = TextSentimentDataset(texts, labels, tokenizer)
dataloader = DataLoader(dataset, batch_size=8, shuffle=True)
print(f"Total data: {len(dataset)} ulasan. Batch size: 8")

## 3. Fine-Tuning Pretrained DistilBERT
Kita muat model `AutoModelForSequenceClassification` dengan 2 label output.

In [ ]:
model_nlp = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(device)

# Gunakan learning rate kecil khusus Transformer (2e-5)
optimizer = optim.AdamW(model_nlp.parameters(), lr=2e-5, weight_decay=0.01)

print("🚀 Memulai Fine-Tuning Transformer (3 Epochs)...")
for epoch in range(1, 4):
    model_nlp.train()
    total_loss, correct, total = 0.0, 0, 0
    
    for batch in dataloader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        batch_labels = batch["label"].to(device)
        
        optimizer.zero_grad()
        outputs = model_nlp(input_ids=input_ids, attention_mask=attention_mask, labels=batch_labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * len(batch_labels)
        preds = outputs.logits.argmax(dim=1)
        correct += (preds == batch_labels).sum().item()
        total += len(batch_labels)
        
    acc = correct / total * 100
    print(f"Epoch {epoch}/3 | Loss: {total_loss/total:.4f} | Accuracy: {acc:.2f}%")

## 4. Live Inference Tester: Uji Model dengan Kalimat Bebas Anda!
Fungsi ini menerima input teks apa saja, melakukan tokenisasi, dan mengembalikan probabilitas sentimen.

In [ ]:
def analyze_review(text):
    model_nlp.eval()
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True).to(device)
    
    with torch.no_grad():
        logits = model_nlp(**inputs).logits
        probs = torch.softmax(logits, dim=1).squeeze().cpu().numpy()
        
    pred_idx = np.argmax(probs)
    sentiment = "POSITIF 🟢" if pred_idx == 1 else "NEGATIF 🔴"
    confidence = probs[pred_idx] * 100
    
    print(f"\n📝 Ulasan: \"{text}\"")
    print(f"Prediksi Sentimen : {sentiment}")
    print(f"Tingkat Keyakinan: {confidence:.2f}%")
    print(f"Probabilitas     : [Negatif: {probs[0]:.3f}, Positif: {probs[1]:.3f}]")

# Uji coba dengan kalimat yang belum pernah dilihat model
analyze_review("The keyboard tactile feedback is incredible and typing feels like magic.")
analyze_review("Worst customer experience ever, broken package and delayed shipping.")